# ML Models Cheat Sheet — Quant Trading Interview Prep

**Structure:**
1. **Setup** — Reusable sklearn evaluation skeleton (preprocessing, GridSearchCV loop, results table)
   - 1.4 **Cross-Validation Strategies** — KFold, Stratified, TimeSeries, GroupKFold, custom panel splits
   - 1.5 **Scoring Metrics** — Regression & classification scorers, custom cost-sensitive scoring
2. **Regression** — 6 models with theory, config dicts & specific methods
3. **Classification** — 4 models, same structure
4. **Unsupervised** — 6 models with adapted evaluation code (no train/test target split)

> **How to use:** Copy the Setup section, then plug in any model config dict from the sections below into `model_configs`. The evaluation loop handles the rest.

---
# 1. Setup — Sklearn Evaluation Skeleton

This skeleton works for **any supervised model** (regression or classification). It handles:
- Feature selection (all features vs. curated subset)
- Preprocessing pipelines (numeric imputation + scaling, categorical imputation + encoding)
- GridSearchCV with proper cross-validation
- Train / CV / Test metrics comparison to diagnose overfit/underfit


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, log_loss,
    silhouette_score, calinski_harabasz_score, davies_bouldin_score
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


### 1.1 Data Preparation & Feature Split

In [ ]:
# ── Adjust these to your dataset ──
target = "heating_energy_kwh_per_sqm"
excluded_cols = ["building_id", "heating_energy_kwh_per_sqm"]

all_features = [col for col in dev_df.columns if col not in excluded_cols]

# Curated feature subset (example: top correlated + engineered + categorical)
top_num = abs(corr).sort_values().tail(6).index.tolist()
selected_features = top_num + ["log_floor_area_sqm"] + ["insulation_grade", "construction_period"]

X_train = dev_df[all_features]
y_train = dev_df[target]
X_test  = test_df[all_features]
y_test  = test_df[target]

# Auto-detect feature types
numeric_features     = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_features_sel     = [f for f in selected_features if f in numeric_features]
categorical_features_sel = [f for f in selected_features if f in categorical_features]


### 1.2 Preprocessing Pipelines

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Preprocessor for ALL features
preprocessor_all = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# Preprocessor for SELECTED features
preprocessor_sel = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features_sel),
    ("cat", categorical_transformer, categorical_features_sel),
])


### 1.3 Evaluation Loop

This is the core loop. Plug any `model_configs` dict from the sections below and run.

**Key design choices:**
- `cv=5` with default KFold. **For time-series data**, replace with `TimeSeriesSplit(n_splits=5)`.
- Scoring: `neg_mean_absolute_error` for regression, swap to `roc_auc` or `neg_log_loss` for classification.
- `mae_over_std` column: if < 0.5, the model captures substantial signal; if > 0.8, barely better than predicting the mean.


In [ ]:
best_estimators = {}
best_feature_sets = {}
results_rows = []

for name, cfg in model_configs.items():
    print(f"Fitting {name}...")
    feat = cfg["features"]
    pipe = Pipeline(steps=[
        ("preprocessor", cfg["preprocessor"]),
        ("model", cfg["model"]),
    ])
    gs = GridSearchCV(
        pipe,
        param_grid=cfg["param_grid"],
        cv=5,                              # <-- swap to TimeSeriesSplit for time-series
        scoring="neg_mean_absolute_error",  # <-- swap for classification
        n_jobs=-1,
    )
    gs.fit(train_df[feat], y_train)

    best = gs.best_estimator_
    best_estimators[name] = best
    best_feature_sets[name] = feat
    best_idx = gs.best_index_

    pred_train = best.predict(train_df[feat])
    pred_test  = best.predict(test_df[feat])

    results_rows.append({
        "model":       name,
        "n_features":  len(feat),
        "best_params": gs.best_params_,
        "train_mae":   round(mean_absolute_error(y_train, pred_train), 1),
        "cv_mae":      round(-gs.cv_results_["mean_test_score"][best_idx], 1),
        "cv_mae_std":  round(gs.cv_results_["std_test_score"][best_idx], 1),
        "test_mae":    round(mean_absolute_error(y_test, pred_test), 1),
        "train_r2":    round(r2_score(y_train, pred_train), 4),
        "test_r2":     round(r2_score(y_test, pred_test), 4),
    })

model_results = pd.DataFrame(results_rows).sort_values("test_mae").reset_index(drop=True)
target_std_test = y_test.std()
model_results["mae_over_std"] = (model_results["test_mae"] / target_std_test).round(4)
display(model_results)


---
### 1.4 Cross-Validation Strategies: Choosing Your Folds

The default `cv=5` in GridSearchCV uses **KFold**, which randomly shuffles and splits data. This is fine for i.i.d. tabular data, but **silently destroys your evaluation** when data has temporal structure, group structure, or both. Choosing the wrong CV strategy is one of the most common sources of **overly optimistic backtests** in quant finance.

**Core principle:** your CV must simulate how the model will be used in production. If in production you predict **new clients**, your val folds must contain unseen clients. If you predict **the future**, your val folds must be strictly in the future. If both — you need both constraints simultaneously.

#### 1.4.1 Standard Strategies

In [ ]:
from sklearn.model_selection import (
    KFold, StratifiedKFold, TimeSeriesSplit,
    GroupKFold, GroupShuffleSplit,
)

# ── KFold (default) — random splits, no structure respected ──
kf = KFold(n_splits=5, shuffle=True, random_state=42)
# Use when: i.i.d. data, no time or group structure
# Danger: if data is sorted by time, consecutive folds will be nearly identical
#         if groups exist (same client in train AND val), you leak information

# ── StratifiedKFold — preserves class distribution. Classification only. ──
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Use when: imbalanced classification (e.g. 5% default rate)
# Why: plain KFold could put 0 defaults in a fold by chance → garbage fold score
# Note: for regression, there's no built-in equivalent, but you can bin the target
#       and stratify on bins

# ── TimeSeriesSplit — expanding window, train always before val ──
tscv = TimeSeriesSplit(n_splits=5)
# Fold 1: train=[0..99],   val=[100..199]
# Fold 2: train=[0..199],  val=[200..299]
# Fold 3: train=[0..299],  val=[300..399]  ... etc.
#
# Use when: single time-series, temporal ordering matters
# Danger: training set grows each fold → early folds have much less data
# Limitation: does NOT handle panels (multiple entities observed over time)

# ── GroupKFold — entire groups stay together (never split across train/val) ──
# ⚠️ KEY MECHANISM — two-step design:
# Step 1: GroupKFold(n_splits=5) knows NOTHING about which column defines groups.
#         It only stores the fold count.
# Step 2: The actual group labels are passed LATER:
#         - Manual loop   → gkf.split(X, y, groups=groups)
#         - GridSearchCV  → gs.fit(X, y, groups=groups)
#           GridSearchCV internally forwards groups to gkf.split() at each fold.
#
# ⚠️ SILENT FAILURE: if you forget groups= in .fit(), sklearn won't raise an error.
#    It falls back to plain KFold — no group logic, silent leakage, inflated scores.

groups = df["client_id"]  # example
gkf = GroupKFold(n_splits=5)

# Manual iteration: groups go to .split()
# for train_idx, val_idx in gkf.split(X, y, groups=groups):
#     assert set(groups[train_idx]) & set(groups[val_idx]) == set()  # no overlap

# With GridSearchCV: groups go to .fit(), NOT to GroupKFold()
# gs = GridSearchCV(pipe, param_grid, cv=gkf, scoring="neg_mean_absolute_error")
# gs.fit(X, y, groups=groups)  # ← groups passed HERE, not at GroupKFold creation
#
# Use when: predictions will be on NEW groups (new clients, new stocks, new stores)
# Key: this is about generalization to unseen entities, not time

#### 1.4.2 Interview Edge — The Two Custom Cases

**Case 1: Panel data, unseen clients in validation (no time constraint)**

**Scenario:** you have monthly data for 500 clients over 3 years. In production, the model will score **new clients** (not seen in training). You need CV folds where each val fold contains clients the model has never seen.

This is exactly **GroupKFold** with `groups=client_id`.

In [ ]:
# ── Case 1: unseen clients ──
groups = df["client_id"]
gkf = GroupKFold(n_splits=5)

# Plug into GridSearchCV — groups MUST be passed to .fit(), not to GroupKFold()
gs = GridSearchCV(pipe, param_grid, cv=gkf, scoring="neg_mean_absolute_error")
gs.fit(X, y, groups=groups)

# Visualization of what's happening:
# Client A: ████████████████  (all months → train)
# Client B: ████████████████  (all months → train)
# Client C: ░░░░░░░░░░░░░░░░  (all months → val)  ← entirely held out
# Client D: ████████████████  (all months → train)
# Client E: ░░░░░░░░░░░░░░░░  (all months → val)  ← entirely held out
#
# Train and val can overlap in TIME — that's fine here.
# The constraint is: no client appears in both train and val.

# Why not just KFold? With KFold, Client C's January data might be in train
# while their March data is in val. The model sees the client's pattern →
# information leaks → inflated score. In production on a truly new client,
# performance will be worse.

**Case 2: Panel data, unseen clients AND future dates**

**Scenario:** same panel, but now in production you'll score **new clients on future dates**. Your CV must enforce **both** constraints simultaneously:
1. Val clients ≠ train clients
2. Val dates > train dates

No sklearn splitter does this out of the box. You need a **custom splitter**:

In [ ]:
# ── Case 2: unseen clients × future dates ──

def panel_temporal_cv(df, group_col, time_col, n_client_splits=3, n_time_splits=3):
    """
    Yields (train_idx, val_idx) where:
    - val contains ONLY clients not in train
    - val dates are strictly AFTER train dates
    
    Approach: 
    1. Split clients into groups (GroupKFold logic)
    2. For each client split, split time (expanding window)
    3. Train = train_clients × past_dates
           Val = val_clients × future_dates
    """
    unique_groups = df[group_col].unique()
    unique_times = np.sort(df[time_col].unique())
    
    # Client splits
    np.random.seed(42)
    np.random.shuffle(unique_groups)
    client_folds = np.array_split(unique_groups, n_client_splits)
    
    # Time splits (expanding window cutoffs)
    time_cutoffs = np.linspace(
        len(unique_times) * 0.4,  # earliest cutoff at 40% of timeline
        len(unique_times) * 0.8,  # latest cutoff at 80%
        n_time_splits
    ).astype(int)
    
    for val_clients in client_folds:
        train_clients = np.setdiff1d(unique_groups, val_clients)
        
        for cutoff_idx in time_cutoffs:
            cutoff_date = unique_times[cutoff_idx]
            
            train_mask = (
                df[group_col].isin(train_clients) & 
                (df[time_col] <= cutoff_date)
            )
            val_mask = (
                df[group_col].isin(val_clients) & 
                (df[time_col] > cutoff_date)
            )
            
            train_idx = df.index[train_mask].values
            val_idx   = df.index[val_mask].values
            
            if len(val_idx) > 0 and len(train_idx) > 0:
                yield train_idx, val_idx


# Usage with GridSearchCV:
cv_splits = list(panel_temporal_cv(
    df, group_col="client_id", time_col="date",
    n_client_splits=3, n_time_splits=3
))

gs = GridSearchCV(
    pipe, param_grid,
    cv=cv_splits,  # <-- list of (train_idx, val_idx) tuples works directly!
    scoring="neg_mean_absolute_error",
)
gs.fit(X, y)

# Visualization:
#              Jan  Feb  Mar  Apr  May  Jun  Jul  Aug  Sep  Oct  Nov  Dec
# Client A:   ████ ████ ████ ████ ████ ████                              ← train (past)
# Client B:   ████ ████ ████ ████ ████ ████                              ← train (past)
# Client C:                                   ░░░░ ░░░░ ░░░░ ░░░░ ░░░░  ← val (future)
# Client D:                                   ░░░░ ░░░░ ░░░░ ░░░░ ░░░░  ← val (future)
#                                        ↑
#                                   time cutoff
#
# Double constraint: val is BOTH unseen clients AND future dates

#### 1.4.3 Decision Guide

```
Is there a time dimension?
├── No  → Is there a group structure (client, stock, store)?
│         ├── No  → KFold (or StratifiedKFold for classification)
│         └── Yes → GroupKFold
└── Yes → Is there a group structure?
          ├── No  → TimeSeriesSplit
          │         (+ consider purge/embargo gap for autocorrelated data)
          └── Yes → Do you need to generalize to BOTH new groups AND future dates?
                    ├── Only new groups    → GroupKFold (time leaks OK if acceptable)
                    ├── Only future dates  → TimeSeriesSplit (group leaks OK)
                    └── Both               → Custom splitter (Case 2 above)
```

#### 1.4.4 Nuances for Interviews

- **GridSearchCV accepts a list of `(train_idx, val_idx)` tuples** as the `cv` argument. This is how you plug any custom split strategy — no need to subclass BaseCrossValidator.
- **Purged CV (López de Prado):** for time-series with autocorrelation, leave a **gap (embargo)** between train and val to prevent leakage from lagged features. Example: if you use 5-day rolling features, purge at least 5 days between train end and val start.
- **Stratified + Time:** sklearn doesn't offer this natively, but you can bin the target and build custom temporal folds that maintain the target distribution — useful for regime-imbalanced financial data.
- **Fewer folds ≠ worse.** With panel temporal CV, you often get 9–15 splits (3 client × 3 time). That's fine — more informative than 5 random KFold splits that leak everywhere.
- **Always count samples per fold.** A custom splitter can silently produce tiny val sets (e.g. 2 new clients × 1 future month = 2 rows). Add a minimum size check.
- **The `groups=` trap:** sklearn's API separates the CV strategy (the splitter object) from the data it operates on (passed at `.split()` or `.fit()` time). For `GroupKFold`, `GroupShuffleSplit`, and `LeaveOneGroupOut`, forgetting `groups=` in `.fit()` silently falls back to non-group behavior. **Always verify** your splits with a quick loop printing `len(train_idx), len(val_idx)` and checking group overlap before running a full grid search.

---
### 1.5 Scoring Metrics: What GridSearchCV Actually Optimizes

`GridSearchCV(scoring=...)` determines which metric drives hyperparameter selection. The default (`None`) uses the model's built-in `.score()` — R² for regression, accuracy for classification — which is **rarely what you want**. Picking the wrong scoring metric means your grid search optimizes for the wrong objective, and the "best" model is best at the wrong thing.

**Core principle:** your scoring metric should reflect your **business loss function**. In quant, a model that's 95% accurate but misses every crash is worthless. A model with low R² but good MAE on extreme values might be exactly what you need for tail risk.

#### 1.5.1 Regression Metrics

All regression scorers in sklearn are **negated** (higher = better convention), so `"neg_mean_absolute_error"` returns −MAE. GridSearchCV maximizes → picks lowest MAE.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# REGRESSION SCORING METRICS — the 3 you need
# ═══════════════════════════════════════════════════════════════

# ── 1. neg_mean_absolute_error ──
# Formula: -1/n * Σ|yi - ŷi|
# The default go-to. Robust to outliers, interpretable in target units.
# "On average, we're off by X kWh / X dollars / X bps."
# Treats all errors equally: underpredicting by 10 = overpredicting by 10.
# Not always true in finance (shorting loss ≠ missed upside).

scoring = "neg_mean_absolute_error"

# ── 2. neg_root_mean_squared_error ──
# Formula: -√(1/n * Σ(yi - ŷi)²)
# Penalizes large errors disproportionately (squaring), but stays in target units
# (unlike raw MSE). Best compromise between outlier sensitivity and interpretability.
# Use when: risk management, VaR, P&L forecasting — anywhere big misses cost more.
# Watch out: sensitive to outliers in the TARGET (one extreme y drags the metric).

scoring = "neg_root_mean_squared_error"

# ── 3. neg_mean_absolute_percentage_error ──
# Formula: -1/n * Σ|yi - ŷi| / |yi|
# Measures RELATIVE error — useful when scale varies across samples.
# Example: forecasting revenue for companies of different sizes (€1M vs €1B).
# ⚠️ EXPLODES when y ≈ 0. Asymmetric: underpredicting 50→100 (100% error)
#    vs overpredicting 100→50 (50% error). Avoid for returns near zero.

scoring = "neg_mean_absolute_percentage_error"

# ── Quick guide ──
# Start with neg_mean_absolute_error (robust, interpretable).
# Switch to neg_root_mean_squared_error if tail accuracy matters.
# Use MAPE only when relative error across different scales is the objective.

#### 1.5.2 Classification Metrics

Some scorers need `predict_proba()`. If your model doesn't support it, GridSearchCV will crash — see 1.5.3 below for fixes.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CLASSIFICATION SCORING METRICS — the 4 you need
# ═══════════════════════════════════════════════════════════════

# ── 1. accuracy ──
# Formula: correct predictions / n
# Use ONLY when classes are balanced (~50/50). Quick sanity check.
# ⚠️ Almost never the right metric in finance. A 95/5 imbalanced dataset
#    gets 95% accuracy by always predicting the majority class.
# Does NOT need predict_proba.

scoring = "accuracy"

# ── 2. roc_auc ──
# Area under the ROC curve. Probability that a random positive ranks
# above a random negative. Threshold-independent.
# The standard in credit scoring, default prediction, alpha signal ranking.
# ⚠️ Needs predict_proba or decision_function.
# ⚠️ Binary only by default. For multiclass: use "roc_auc_ovr" or "roc_auc_ovo".

scoring = "roc_auc"

# ── 3. average_precision ──
# Area under the Precision-Recall curve. Focuses on positive class performance.
# THE metric for imbalanced data where you care about the rare class
# (fraud, default, earnings surprise). Much more informative than AUC
# when positives are < 10% — AUC can look great while precision is garbage.
# ⚠️ Needs predict_proba.

scoring = "average_precision"

# ── 4. neg_log_loss ──
# Formula: -1/n * Σ[y·log(p̂) + (1-y)·log(1-p̂)]
# Measures probability CALIBRATION quality, not just ranking.
# Use when downstream decisions consume P(event) directly:
# Kelly criterion sizing, risk budgeting, insurance pricing.
# Heavily penalizes confident wrong predictions (p̂=0.99 when y=0 → huge loss).
# ⚠️ Needs predict_proba.

scoring = "neg_log_loss"

# ── Quick guide ──
# Default: roc_auc (threshold-free ranking quality).
# Rare events (<5% positive): average_precision.
# Need calibrated probabilities: neg_log_loss.
# Balanced sanity check only: accuracy.

#### 1.5.3 Model Requirements: What Can Go Wrong

Some scoring metrics need `predict_proba()`, which not all models provide. And class imbalance is handled by the **model**, not the scorer — these are independent choices.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# WHICH MODELS SUPPORT predict_proba?
# ═══════════════════════════════════════════════════════════════

# ── Models that ALWAYS have predict_proba ──
# LogisticRegression, RandomForestClassifier, GradientBoostingClassifier,
# GaussianNB, XGBClassifier, LGBMClassifier
# → roc_auc, average_precision, neg_log_loss all work out of the box.

# ── Models that DON'T — and how to fix them ──

# SVC: add probability=True (fits Platt scaling internally, makes training slower)
from sklearn.svm import SVC
svc = SVC(kernel="rbf", probability=True)  # now predict_proba works

# LinearSVC: no probability param → wrap in CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
calibrated_svc = CalibratedClassifierCV(LinearSVC(C=1.0), cv=5)
# calibrated_svc.predict_proba() now works

# SGDClassifier: depends on the loss function
from sklearn.linear_model import SGDClassifier
sgd = SGDClassifier(loss="log_loss")  # log_loss → has predict_proba
# sgd = SGDClassifier(loss="hinge")  # hinge → NO predict_proba (SVM-style)

# ═══════════════════════════════════════════════════════════════
# CLASS IMBALANCE: MODEL SIDE vs SCORER SIDE
# ═══════════════════════════════════════════════════════════════

# The scorer MEASURES performance; the model HANDLES imbalance.
# These are INDEPENDENT choices. Don't confuse them.

# Option 1: class_weight in the model (reweights the training loss)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
LogisticRegression(class_weight="balanced")    # auto-adjusts by inverse frequency
RandomForestClassifier(class_weight="balanced")
SVC(class_weight="balanced")

# Option 2: XGBoost / LightGBM specific
# XGBClassifier(scale_pos_weight=neg_count / pos_count)
# LGBMClassifier(is_unbalance=True)

# Option 3: DON'T touch the model, just use the right scorer.
# With roc_auc or average_precision, imbalance is already accounted for
# in the metric itself. Let the model learn the true distribution,
# then evaluate with an imbalance-aware metric. This is often cleaner.

# ⚠️ COMMON MISTAKE: class_weight="balanced" + scoring="accuracy"
# The model upweights the rare class → predicts more positives → accuracy drops.
# You "fixed" imbalance in the model but measure with a metric that doesn't care.
# Fix: pair class_weight="balanced" with roc_auc or average_precision.

#### 1.5.4 Custom Scorer: Cost-Sensitive FP/FN

When false positives and false negatives have **different business costs**, no built-in metric captures your objective. Example: predicting loan defaults where missing a default (FN) costs €50k (loss on the loan) but wrongly rejecting a good client (FP) costs €2k (lost revenue).

`make_scorer` is the bridge between any Python function and GridSearchCV. Know its three key arguments:
- The scoring function itself
- `greater_is_better` (default `True`)
- `needs_proba` (default `False` → receives `predict` output; `True` → receives `predict_proba[:, 1]`)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CUSTOM COST-SENSITIVE SCORER
# ═══════════════════════════════════════════════════════════════

from sklearn.metrics import make_scorer, confusion_matrix

# ── Version 1: hard predictions (uses predict at threshold 0.5) ──

def cost_sensitive_score(y_true, y_pred, cost_fp=2_000, cost_fn=50_000):
    """
    Returns NEGATIVE total cost (because GridSearchCV maximizes).
    Lower total cost → higher (less negative) score → GridSearchCV picks it.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_cost = (fp * cost_fp) + (fn * cost_fn)
    return -total_cost

cost_scorer = make_scorer(
    cost_sensitive_score,
    cost_fp=2_000,
    cost_fn=50_000,
    # greater_is_better=True is the default — our function already returns negative
)

# Usage: exactly like a built-in scorer
# gs = GridSearchCV(pipe, param_grid, cv=5, scoring=cost_scorer)
# gs.fit(X, y)


# ── Version 2: with custom threshold on probabilities ──
# Version 1 uses the default 0.5 threshold. But if FN costs 25× more than FP,
# you probably want a LOWER threshold (predict default more aggressively).

def cost_at_threshold(y_true, y_proba, threshold=0.3, cost_fp=2_000, cost_fn=50_000):
    """Cost-sensitive score with custom decision threshold."""
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return -(fp * cost_fp + fn * cost_fn)

cost_scorer_proba = make_scorer(
    cost_at_threshold,
    needs_proba=True,     # ← sklearn passes predict_proba[:, 1] instead of predict
    threshold=0.3,        # ← lower = more conservative = fewer missed defaults
    cost_fp=2_000,
    cost_fn=50_000,
)

# Usage:
# gs = GridSearchCV(pipe, param_grid, cv=5, scoring=cost_scorer_proba)
# gs.fit(X, y)
# ⚠️ needs_proba=True → model MUST have predict_proba (see 1.5.3)

#### 1.5.5 Nuances for Interviews

- **Threshold tuning ≠ model tuning.** Hyperparameters control what the model learns; the threshold controls the decision boundary on the learned probabilities. Optimize them separately: GridSearchCV for hyperparameters, then a sweep on the val set for threshold.
- **`roc_auc` is threshold-free; `accuracy` is not.** If an interviewer asks "which metric?", saying `roc_auc` first shows you understand that model quality and decision threshold are separate concerns.
- **Multi-metric GridSearchCV** is possible with `scoring={"auc": "roc_auc", "ap": "average_precision"}` and `refit="auc"`. Tracks multiple metrics in one grid search. `refit` specifies which metric selects the best model.
- **In quant, the real loss function is P&L.** If you can express your trading strategy's P&L as a function of predictions, build a custom scorer around it. "I optimized my model for Sharpe ratio of the strategy" is a much stronger interview answer than "I used roc_auc."

---
# 2. Regression Models


## 2.1 Linear Regression Family — Ridge, Lasso, ElasticNet

### How It Works
Ordinary Least Squares (OLS) minimizes the sum of squared residuals: $\min_w ||y - Xw||^2_2$.  
The problem: with many features or correlated features (multicollinearity), OLS overfits and coefficients blow up.

Regularization fixes this by adding a penalty term to the loss:
- **Ridge (L2):** $||y - Xw||^2 + \alpha ||w||^2_2$ — shrinks all coefficients toward zero but never exactly to zero. Closed-form solution: $w = (X^TX + \alpha I)^{-1}X^Ty$.
- **Lasso (L1):** $||y - Xw||^2 + \alpha ||w||_1$ — drives some coefficients exactly to zero → **automatic feature selection**. No closed form; solved via coordinate descent.
- **ElasticNet:** $||y - Xw||^2 + \alpha(\rho||w||_1 + (1-\rho)||w||^2_2)$ — combines both. The L2 part handles correlated features that Lasso can't (Lasso arbitrarily picks one from a correlated group).

### When to Use / What Good Performance Means
- **Baseline for every regression task.** If Ridge/Lasso works well, your signal is approximately linear — don't add unnecessary complexity.
- **Ridge >> OLS** → multicollinearity was inflating your coefficients. Very common with financial features (e.g., correlated macro indicators).
- **Lasso >> Ridge** → you had genuinely sparse signal (only a few features actually matter). Typical in factor research with hundreds of candidate alphas.
- **ElasticNet >> Lasso** → you have groups of correlated useful features. Lasso would arbitrarily keep one; ElasticNet keeps the group.

### Interview Edge — Quant Trading
- **Lasso limitation:** Cannot select more features than samples (when p > n). ElasticNet fixes this — critical for genomics-style or wide financial datasets.
- **Standardize features BEFORE regularization.** Otherwise the penalty is scale-dependent: a feature in dollars gets penalized differently than one in percentages. This is a common interview trap.
- **In finance:** Ridge is almost always better than OLS because financial features are correlated. Lasso is the go-to for factor selection in alpha research.
- **Bias-variance tradeoff:** Ridge coefficients are biased (shrunk toward zero) but have lower variance. Know the math: $\text{MSE} = \text{Bias}^2 + \text{Variance}$. Adding a little bias can dramatically reduce variance.
- **Ridge never zeros out coefficients.** If you need to explain which features matter (interpretability for PM or risk team), use Lasso or ElasticNet.
- **Degrees of freedom:** Ridge's effective degrees of freedom = $\sum_i \frac{d_i^2}{d_i^2 + \alpha}$ where $d_i$ are singular values of X. This is a favorite interview formula.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `alpha` (λ) | Regularization strength. Higher → more shrinkage → simpler | 1e-4 to 1e+3 (log scale). Use `RidgeCV`/`LassoCV` | In finance with noisy data, optimal λ is often surprisingly high. For Lasso, very high α zeros out ALL features — always check |
| `l1_ratio` (ElasticNet) | Blend: 0 = pure Ridge, 1 = pure Lasso | 0.1 to 0.9, step 0.1. Sweet spot: 0.5–0.7 for sparse financial signals | If optimal → 1, just use Lasso. If → 0, just use Ridge |


In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

model_configs = {
    # ── Ridge ──
    "ridge_all": {
        "model": Ridge(),
        "param_grid": {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]},
        "features": all_features,
        "preprocessor": preprocessor_all,
    },
    "ridge_top": {
        "model": Ridge(),
        "param_grid": {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]},
        "features": selected_features,
        "preprocessor": preprocessor_sel,
    },
    # ── Lasso ──
    "lasso_all": {
        "model": Lasso(max_iter=10000),
        "param_grid": {"model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]},
        "features": all_features,
        "preprocessor": preprocessor_all
    },
    # ── ElasticNet ──
    "elasticnet_all": {
        "model": ElasticNet(max_iter=10000),
        "param_grid": {
            "model__alpha": [0.01, 0.1, 1.0, 10.0],
            "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    }
}


In [ ]:
# ── Specific Methods to Know ──

# 1. Coefficients & feature importance
best_ridge = best_estimators["ridge_all"]
feature_names = best_ridge.named_steps["preprocessor"].get_feature_names_out()
coefs = pd.Series(best_ridge.named_steps["model"].coef_, index=feature_names)
print("Top 10 features by |coefficient|:")
print(coefs.abs().sort_values(ascending=False).head(10))

# 2. Lasso: which features were zeroed out?
best_lasso = best_estimators["lasso_all"]
lasso_coefs = best_lasso.named_steps["model"].coef_
n_zero = (lasso_coefs == 0).sum()
print(f"\nLasso zeroed out {n_zero}/{len(lasso_coefs)} features")

# 3. Auto alpha selection (faster than GridSearchCV)
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5)
lasso_cv = LassoCV(alphas=np.logspace(-4, 2, 50), cv=5, max_iter=10000)
# ridge_cv.fit(X_preprocessed, y_train)
# print(f"Best alpha: {ridge_cv.alpha_}")

# 4. Regularization path (visualize coefficient shrinkage)
# from sklearn.linear_model import lasso_path
# alphas, coefs_path, _ = lasso_path(X_preprocessed, y_train, alphas=np.logspace(-4, 1, 50))
# # Plot coefs_path vs alphas to see when features drop out


## 2.2 Decision Tree Regression

### How It Works
Recursively partitions the feature space into rectangular regions by choosing the split (feature + threshold) that minimizes the MSE in the resulting child nodes. Each leaf predicts the **mean** of training samples that fall in it.

The algorithm is greedy: at each node, it picks the locally optimal split without considering downstream consequences. This makes it fast but potentially suboptimal (you can't backtrack to fix an early bad split).

### When to Use / What Good Performance Means
- Captures **non-linear relationships and feature interactions** without manual feature engineering (e.g., "if temperature > 30 AND humidity > 80% → energy spikes").
- If a single tree beats linear models significantly → strong non-linearities or interaction effects exist in your data.
- **Rarely used alone in practice** — serves as the building block for Random Forest and Gradient Boosting. Understanding it deeply is key to understanding ensembles.

### Interview Edge — Quant Trading
- **Cannot extrapolate:** predictions are always bounded by the training target range. If your model sees prices up to 100 in training, it will never predict 101. Critical for financial time-series with trends or regime shifts.
- **High instability:** a small change in training data can produce a completely different tree. This instability is exactly WHY bagging works — averaging many unstable estimators reduces variance.
- **Feature importance from impurity (Gini/MSE reduction) is biased** toward high-cardinality features (e.g., a continuous feature gets more chances to split than a binary one). Use **permutation importance** instead.
- **In quant:** a deep tree with high training accuracy almost certainly overfits noise. Always compare against a pruned/shallow version. The gap between train and test performance is your overfit diagnostic.
- **Axis-aligned splits:** trees can only split parallel to feature axes. They need many splits to approximate a diagonal boundary. This is why linear models can beat trees on truly linear problems.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `max_depth` | Primary overfit control. Deeper = more complex | 3–10 standalone; 2–6 as boosting base learner | In noisy financial data, rarely go beyond 5–6 |
| `min_samples_leaf` | Min observations per leaf. Smoother predictions | 5–50 (small data), 20–200+ (large) | More robust than max_depth for heterogeneous data. In finance, err high (50+) |
| `min_samples_split` | Min samples to attempt a split | 2 (default)–50 | Less commonly tuned; min_samples_leaf is more intuitive |
| `max_features` | Subset of features to consider per split | None (all) for single tree. sqrt/log2 for ensembles | Only matters in ensemble context; single tree usually uses all |


In [ ]:
from sklearn.tree import DecisionTreeRegressor

model_configs = {
    "dtree_all": {
        "model": DecisionTreeRegressor(random_state=42),
        "param_grid": {
            "model__max_depth": [3, 5, 7, 10, None],
            "model__min_samples_leaf": [5, 10, 20, 50]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    }
}


In [ ]:
# ── Specific Methods to Know ──

# 1. Visualize the tree (essential for understanding splits)
from sklearn.tree import plot_tree, export_text

best_dt = best_estimators["dtree_all"]
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(best_dt.named_steps["model"], max_depth=3, filled=True,
          feature_names=best_dt.named_steps["preprocessor"].get_feature_names_out(),
          ax=ax, fontsize=8)
plt.title("Decision Tree (top 3 levels)")
plt.tight_layout()
plt.show()

# 2. Text representation
print(export_text(best_dt.named_steps["model"], max_depth=3,
                  feature_names=list(best_dt.named_steps["preprocessor"].get_feature_names_out())))

# 3. Feature importance (impurity-based — biased but fast)
importances = best_dt.named_steps["model"].feature_importances_
feat_names = best_dt.named_steps["preprocessor"].get_feature_names_out()
pd.Series(importances, index=feat_names).sort_values(ascending=False).head(10).plot.barh()
plt.title("Impurity-Based Feature Importance")
plt.show()

# 4. Cost-complexity pruning path (find optimal alpha for pruning)
# dt = DecisionTreeRegressor(random_state=42)
# path = dt.cost_complexity_pruning_path(X_train_processed, y_train)
# ccp_alphas = path.ccp_alphas  # use CV to pick optimal ccp_alpha


## 2.3 Random Forest Regression

### How It Works
An ensemble of B decision trees, each trained on a **bootstrap sample** (random sample with replacement) of the data. At each split, only a random subset of features is considered. The final prediction is the **average** of all trees' predictions.

Two sources of randomness decorrelate the trees:
1. **Bagging** (bootstrap aggregating): each tree sees a different ~63% of the data
2. **Feature subsampling**: each split considers only `max_features` features

This decorrelation is the key insight: averaging correlated estimators barely reduces variance; averaging decorrelated ones reduces it dramatically.

### When to Use / What Good Performance Means
- **Strong general-purpose model** that works well out-of-the-box with minimal tuning.
- **RF >> single tree** → variance reduction via averaging is helping (your data is noisy but the signal is there).
- **RF ≈ linear model** → your data is mostly linear; RF adds unnecessary complexity.
- **RF << GBM** → your data benefits from bias reduction (boosting's strength), not just variance reduction.

### Interview Edge — Quant Trading
- **RF reduces variance but NOT bias.** If individual trees underfit (too shallow, too few features), the forest will underfit too. You can't average your way out of underfitting.
- **Cannot extrapolate:** predictions are bounded by the training target range (average of trees that are each bounded). Problematic for trending financial time-series.
- **OOB (out-of-bag) error:** ~37% of data is excluded from each tree's bootstrap → free validation. But **NOT valid for time-series** (OOB samples can be from the future → lookahead bias).
- **Feature importance caveats:** impurity-based importance is biased toward continuous/high-cardinality features. **Permutation importance** or **SHAP** is the gold standard.
- **In quant:** excellent for cross-sectional predictions (ranking stocks at a point in time), less useful for pure time-series forecasting.
- **Variance of RF predictions** across trees can be used as an uncertainty estimate — useful for position sizing.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `n_estimators` | Number of trees. More = less variance, diminishing returns | 100–1000. Start 200 | RF does NOT overfit with more trees (unlike GBM). More is weakly better, just slower |
| `max_features` | Features per split. Lower = more decorrelation | `sqrt(p)` classification, `p/3` or `1.0` regression. Try 0.3–0.7 | **Key lever:** lower → more bias, less tree correlation → often better overall |
| `max_depth` | Tree depth. None = fully grown (default) | None (default), or 10–30 if overfitting | Combined with min_samples_leaf for effective complexity control |
| `min_samples_leaf` | Minimum samples per leaf | 1–20. In finance: 5–50 | Smooths predictions. Higher = more regularization |
| `n_jobs` | Parallelism. -1 = all cores | -1 | Trees are embarrassingly parallel — always use -1 |


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_configs = {
    "rf_all": {
        "model": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
        "param_grid": {
            "model__max_depth": [8, 12, 20, None],
            "model__max_features": [0.3, 0.5, 0.7, 1.0],
            "model__min_samples_leaf": [1, 5, 10, 20]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    }
}


In [ ]:
# ── Specific Methods to Know ──

# 1. OOB score (free validation — do NOT use for time-series!)
rf_oob = RandomForestRegressor(n_estimators=300, oob_score=True, random_state=42, n_jobs=-1)
# rf_oob.fit(X_train_processed, y_train)
# print(f"OOB R2: {rf_oob.oob_score_:.4f}")

# 2. Permutation importance (unbiased, model-agnostic)
from sklearn.inspection import permutation_importance
best_rf = best_estimators["rf_all"]
# perm_imp = permutation_importance(best_rf, test_df[all_features], y_test,
#                                    n_repeats=10, random_state=42, n_jobs=-1)
# pd.Series(perm_imp.importances_mean, index=all_features).sort_values(ascending=False).head(10)

# 3. Prediction variance across trees (uncertainty estimate)
# individual_preds = np.array([tree.predict(X_test_processed) for tree in best_rf.named_steps["model"].estimators_])
# pred_mean = individual_preds.mean(axis=0)
# pred_std  = individual_preds.std(axis=0)  # <-- uncertainty per prediction

# 4. Partial dependence plots (marginal effect of a feature)
from sklearn.inspection import PartialDependenceDisplay
# PartialDependenceDisplay.from_estimator(best_rf, test_df[all_features], features=[0, 1], ax=ax)


# 5. Learning curve: score vs number of trees (warm_start + OOB)
rf_warm = RandomForestRegressor(
    n_estimators=1, warm_start=True, oob_score=True,
    random_state=42, max_depth=12, max_features=0.5, n_jobs=-1
)

n_tree_steps = list(range(10, 510, 10))  # 10, 20, ..., 500
warm_results = []

for n_trees in n_tree_steps:
    rf_warm.n_estimators = n_trees
    rf_warm.fit(X_train_processed, y_train)

    pred_train = rf_warm.predict(X_train_processed)
    pred_test  = rf_warm.predict(X_test_processed)

    warm_results.append({
        "n_trees":   n_trees,
        "train_mae": round(mean_absolute_error(y_train, pred_train), 2),
        "test_mae":  round(mean_absolute_error(y_test, pred_test), 2),
        "oob_r2":    round(rf_warm.oob_score_, 4),  # R² on out-of-bag samples
    })

warm_df = pd.DataFrame(warm_results)
display(warm_df)

# Plot convergence: MAE + OOB R²
fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.plot(warm_df["n_trees"], warm_df["train_mae"], label="Train MAE", color="tab:blue")
ax1.plot(warm_df["n_trees"], warm_df["test_mae"],  label="Test MAE",  color="tab:orange")
ax1.set_xlabel("Number of trees"); ax1.set_ylabel("MAE")
ax1.legend(loc="center left")

ax2 = ax1.twinx()
ax2.plot(warm_df["n_trees"], warm_df["oob_r2"], label="OOB R²", color="tab:green", linestyle="--")
ax2.set_ylabel("OOB R²")
ax2.legend(loc="center right")

plt.title("RF convergence: MAE & OOB R² vs n_estimators (warm_start)")
plt.tight_layout(); plt.show()

# Notes:
# - oob_score_ is R² by default for regressors (variance explained on ~37% OOB samples)
# - With few trees (<20-30), OOB is unstable (some samples OOB for only 1-2 trees)
# - ⚠️ OOB is NOT valid for time-series data (OOB samples can be from the future)
# - RF does NOT overfit with more trees: test MAE plateaus (unlike GBM)
# - The plateau tells you the useful budget (often 200-400 trees suffice)

## 2.4 Gradient Boosting — XGBoost / LightGBM / sklearn

### How It Works
Sequentially fits **weak learners** (shallow trees) to the **negative gradient** (residuals for MSE) of the loss function. Each new tree corrects the errors remaining after all previous trees. Final prediction = initial prediction + learning_rate × Σ(tree predictions).

**Key differences from RF:**
- RF: parallel, independent trees → reduces variance
- GBM: sequential, dependent trees → reduces bias
- RF: can't overfit with more trees; GBM: CAN overfit with too many rounds

**XGBoost vs LightGBM:**
- XGBoost: level-wise tree growth (balanced), more conservative, slightly slower
- LightGBM: leaf-wise growth (grows the leaf with highest loss reduction), faster, can overfit more on small data
- Both add L1/L2 regularization on leaf weights (unlike vanilla gradient boosting)

### When to Use / What Good Performance Means
- **State-of-the-art for tabular data.** Wins most Kaggle competitions and is dominant in production quant systems.
- **GBM >> RF** → your data benefits from bias reduction. GBM can learn complex patterns that RF can't capture.
- **GBM ≈ RF** → both capture the signal well; prefer RF for simplicity and stability.

### Interview Edge — Quant Trading
- **Early stopping is the single most important "hyperparameter".** Train with a validation set and stop when val loss stops improving. This prevents overfitting far more reliably than tuning n_estimators manually.
- **learning_rate and n_estimators are coupled:** halve the learning rate, double the trees. There's a sweet spot where this rule breaks down (diminishing returns).
- **Gradient boosting is fitting the GRADIENT, not the residuals** (they coincide for MSE but differ for other losses). Know this for interviews — it's the actual generalization.
- **XGBoost's second-order approximation** uses both gradient AND Hessian of the loss → faster convergence than first-order methods. This is why XGBoost outperforms sklearn's GBM.
- **In quant:** GBM classifiers dominate cross-sectional alpha (stock selection). Use purged/embargo CV (López de Prado) to avoid lookahead bias.
- **Stacking:** GBM on top of simpler model predictions is a common interview topic and practical technique.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `learning_rate` (eta) | Shrinks each tree's contribution | 0.01–0.3. Finance: 0.01–0.05 | Lower = more trees needed but better generalization |
| `n_estimators` | Boosting rounds. Too many → overfit | 100–5000+. ALWAYS use early stopping | Never tune in isolation — depends on learning_rate |
| `max_depth` | Interaction order. Depth d = d-way interactions | 3–8. Finance: 3–5 | LightGBM uses `num_leaves` instead (31 default ≈ depth 5) |
| `subsample` | Row subsampling per tree | 0.6–0.9. Noisy data: 0.5–0.7 | Adds RF-like randomness → reduces overfit |
| `colsample_bytree` | Feature subsampling per tree | 0.6–0.9 | Same effect as max_features in RF |
| `reg_alpha` / `reg_lambda` | L1/L2 on leaf weights | 0–10. Usually left at default | Less impactful than above params; use as fine-tuning |


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
# For production, use XGBoost or LightGBM directly:
# from xgboost import XGBRegressor
# from lightgbm import LGBMRegressor

model_configs = {
    "gbm_all": {
        "model": GradientBoostingRegressor(
            random_state=42,
            n_iter_no_change=20,       # stop if no improvement for 20 consecutive rounds
            validation_fraction=0.15,  # 15% of train held out to monitor convergence
            tol=1e-4,                  # minimum improvement threshold
        ),
        "param_grid": {
            "model__n_estimators": [500, 1000, 2000],    # set high — early stopping cuts
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [3, 5, 7],
            "model__subsample": [0.7, 0.9],
            "model__min_samples_leaf": [5, 10, 20]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    }
}
# n_iter_no_change logic:
# - Reserves validation_fraction of training data as internal validation set
# - After each boosting round, checks if val score improved by at least tol
# - If no improvement for n_iter_no_change consecutive rounds → training stops
# - After fit: best_estimator_.named_steps["model"].n_estimators_ = actual rounds used
# - This is the sklearn equivalent of XGBoost's early_stopping_rounds
#
# Why set n_estimators high?
# - You're letting early stopping decide when to stop, not a fixed budget
# - n_estimators becomes a CEILING, not a target
# - In GBM, unlike RF, more rounds CAN overfit — early stopping prevents this

# ── XGBoost / LightGBM configs (preferred in practice) ──
# model_configs["xgb_all"] = {
#     "model": XGBRegressor(n_estimators=1000, learning_rate=0.05, random_state=42,
#                            tree_method="hist", early_stopping_rounds=50),
#     "param_grid": {
#         "model__max_depth": [3, 5, 7],
#         "model__subsample": [0.7, 0.8],
#         "model__colsample_bytree": [0.7, 0.8],
#         "model__reg_alpha": [0, 1],
#         "model__reg_lambda": [1, 5],
#     },
#     "features": all_features,
#     "preprocessor": preprocessor_all,
# }

In [ ]:
# ── Specific Methods to Know ──

# 1. Early stopping (XGBoost/LightGBM — the MOST important technique)
# xgb = XGBRegressor(n_estimators=5000, learning_rate=0.01, early_stopping_rounds=50)
# xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
# print(f"Best iteration: {xgb.best_iteration}")

# 2. Staged predictions (sklearn GBM — see train vs val loss over boosting rounds)
best_gbm = best_estimators.get("gbm_all")
# staged_preds = list(best_gbm.named_steps["model"].staged_predict(X_test_processed))
# staged_mae = [mean_absolute_error(y_test, sp) for sp in staged_preds]
# plt.plot(staged_mae)
# plt.xlabel("Boosting Round"); plt.ylabel("Test MAE"); plt.title("Learning Curve")

# 3. SHAP values (gold standard for interpretability)
# import shap
# explainer = shap.TreeExplainer(best_gbm.named_steps["model"])
# shap_values = explainer.shap_values(X_test_processed)
# shap.summary_plot(shap_values, X_test_processed, feature_names=feature_names)

# 4. Feature importance comparison
# xgb_importance types: 'weight' (split count), 'gain' (avg gain), 'cover' (avg samples)
# 'gain' is most informative; 'weight' is biased toward high-cardinality features


## 2.5 SVR (Support Vector Regression)

### How It Works
Finds a function that fits the data within an **ε-insensitive tube**: errors smaller than ε are ignored (zero loss), only errors exceeding ε are penalized. The model maximizes the "flatness" of the fitting function while keeping most points inside the tube.

With kernels (RBF, polynomial), the data is implicitly mapped to a high-dimensional space where a linear ε-tube is fit. The **kernel trick** computes inner products in this space without ever computing the actual transformation: $K(x, x') = \phi(x)^T\phi(x')$.

Only the training points that lie ON or OUTSIDE the tube boundary (the **support vectors**) determine the model — all other points could be moved freely without changing the result.

### When to Use / What Good Performance Means
- Works well on **small-to-medium datasets** (< 50k samples) with clear non-linear structure.
- **SVR with RBF >> linear SVR** → strong non-linearity. **Linear SVR ≈ RBF** → data is mostly linear.
- Less common in production quant (poor scalability), but frequently asked about in interviews.

### Interview Edge — Quant Trading
- **ε-insensitive loss makes SVR robust to small noise** — it only penalizes errors beyond ε. This is a natural fit for noisy financial data where you don't want to chase every small fluctuation.
- **Kernel choice IS the model choice:** linear SVR ≈ Ridge with hinge loss; RBF SVR is a universal approximator.
- **Doesn't scale:** O(n²) to O(n³) training. Impractical for > 50k samples without approximations (Nyström, random Fourier features).
- **Support vectors = the only data points that matter.** The model is sparse in data space. Classic interview question: "What are support vectors and why do they matter?"
- **In quant:** occasionally used for small-sample regime detection or non-linear factor models. Also for options pricing surface fitting.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `C` | Regularization. Higher = less tolerance for violations = tighter fit | 0.01–1000 (log scale) | C and ε interact: high C + small ε = overfit |
| `epsilon` (ε) | Width of insensitive tube. Larger = more errors tolerated | 0.01–0.5. Scale relative to target variance | Set proportional to noise level. In finance: larger ε |
| `kernel` | Feature mapping. RBF, linear, poly, sigmoid | Try linear first, then RBF | RBF with proper gamma is a universal approximator |
| `gamma` (RBF) | Inverse of influence radius. High = spiky = overfit | 'scale' default. Search 1e-4 to 1e+1 | gamma too high = each point creates its own peak = memorization |


In [ ]:
from sklearn.svm import SVR

model_configs = {
    "svr_rbf_all": {
        "model": SVR(kernel="rbf"),
        "param_grid": {
            "model__C": [0.1, 1.0, 10.0, 100.0],
            "model__epsilon": [0.01, 0.05, 0.1, 0.2],
            "model__gamma": ["scale", 0.001, 0.01, 0.1]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    },
    "svr_linear_all": {
        "model": SVR(kernel="linear"),
        "param_grid": {
            "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
            "model__epsilon": [0.01, 0.05, 0.1]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    }
}


In [ ]:
# ── Specific Methods to Know ──

# 1. Support vectors: which training points define the model
best_svr = best_estimators.get("svr_rbf_all")
# sv_indices = best_svr.named_steps["model"].support_
# n_sv = len(sv_indices)
# print(f"Number of support vectors: {n_sv} / {len(X_train)} ({n_sv/len(X_train)*100:.1f}%)")
# A high % of support vectors means most data is near the boundary → complex problem

# 2. Dual coefficients (the alphas in the dual formulation)
# dual_coefs = best_svr.named_steps["model"].dual_coef_

# 3. For large datasets: use LinearSVR (optimized for linear kernel, scales to millions)
from sklearn.svm import LinearSVR
# linear_svr = LinearSVR(C=1.0, epsilon=0.1, max_iter=10000)

# 4. Scaling is CRITICAL for SVR — always standardize features
# The preprocessor pipeline above handles this, but if doing manually:
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X_train)


## 2.6 KNN Regression (K-Nearest Neighbors)

### How It Works
For each test point, finds the K closest training points in feature space and predicts the **average** (or distance-weighted average) of their target values. No explicit training phase — the model IS the training data ("lazy learning").

Distance is typically Euclidean, but Manhattan, Minkowski, or Mahalanobis can be used. The choice of distance metric implicitly defines your similarity measure.

### When to Use / What Good Performance Means
- Works when the **signal is local**: nearby points in feature space have similar targets.
- If KNN works well → strong local structure exists. Common in spatial or cross-sectional financial data.
- Often used **as a feature** (distance to K-th neighbor, or average target of K neighbors) rather than as the final model.

### Interview Edge — Quant Trading
- **Curse of dimensionality:** in high dimensions, all points are roughly equidistant → KNN breaks down. Need D < ~15–20 effective features. Reduce dimensions first (PCA).
- **Distance metric choice is critical.** Euclidean assumes isotropic features → **standardize first**. Manhattan is more robust to outliers.
- **No model to inspect** — you can't extract "what the model learned" (unlike coefficients or trees). Interpretability comes only from analyzing neighbors.
- **In quant:** used in pairs trading (find historically similar market conditions), and as features in stacking ensembles.
- **Computational cost at prediction time:** O(n·d) per query without data structures, O(d·log n) with KD-trees. Matters for real-time prediction.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `n_neighbors` (K) | Small K = noisy; large K = smooth, may underfit | 3–50. sqrt(n) rough start. Use CV | In finance: 20–50 usually better (noise). Too large → mean reversion |
| `weights` | 'uniform' = equal weight; 'distance' = inverse-distance weighted | Try both. Distance-weighted usually better | Distance weighting partially mitigates too-large K |
| `metric` | Distance function | Euclidean, Manhattan, Minkowski, Mahalanobis | In finance: Mahalanobis handles correlated features. Or PCA + Euclidean |
| `algorithm` | Search algorithm | 'auto', 'ball_tree', 'kd_tree', 'brute' | 'auto' is fine. KD-tree fails in high-D (>20 features) |


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

model_configs = {
    "knn_all": {
        "model": KNeighborsRegressor(),
        "param_grid": {
            "model__n_neighbors": [5, 10, 20, 30, 50],
            "model__weights": ["uniform", "distance"],
            "model__metric": ["euclidean", "manhattan"]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    }
}


In [ ]:
# ── Specific Methods to Know ──

# 1. Retrieve actual neighbors (useful for debugging & understanding)
best_knn = best_estimators.get("knn_all")
# distances, indices = best_knn.named_steps["model"].kneighbors(X_test_processed[:5])
# For each test point, shows which training points it's using and how far they are

# 2. KNN as a FEATURE (common in stacking / feature engineering)
# For each point, compute: avg target of K neighbors, distance to K-th neighbor, etc.
# knn_feat = KNeighborsRegressor(n_neighbors=10, weights="distance")
# knn_feat.fit(X_train_processed, y_train)
# knn_distance_feature = knn_feat.kneighbors(X_train_processed)[0].mean(axis=1)

# 3. Dimensionality reduction BEFORE KNN (critical for high-D)
from sklearn.decomposition import PCA
# pca = PCA(n_components=10)
# X_reduced = pca.fit_transform(X_train_processed)
# knn_on_pca = KNeighborsRegressor(n_neighbors=20).fit(X_reduced, y_train)

# 4. Validation curve: visualize K vs error
# from sklearn.model_selection import validation_curve
# k_range = np.arange(1, 51)
# train_scores, val_scores = validation_curve(
#     KNeighborsRegressor(), X_processed, y_train,
#     param_name="n_neighbors", param_range=k_range, cv=5,
#     scoring="neg_mean_absolute_error"
# )


---
# 3. Classification Models

> **Scoring change:** For classification, swap `scoring="neg_mean_absolute_error"` in the evaluation loop to:
> - `"roc_auc"` — for binary classification (probability ranking quality)
> - `"neg_log_loss"` — for calibrated probability quality
> - `"accuracy"` — only if classes are balanced
>
> Also replace MAE/R² metrics with `accuracy_score`, `roc_auc_score`, `classification_report`, `confusion_matrix`.


In [ ]:
# ── Adapted evaluation loop for classification ──
# Replace the metrics collection block in the main loop with:

# pred_train = best.predict(train_df[feat])
# pred_test  = best.predict(test_df[feat])
# proba_test = best.predict_proba(test_df[feat])[:, 1]  # for binary
#
# results_rows.append({
#     "model":       name,
#     "best_params": gs.best_params_,
#     "train_acc":   round(accuracy_score(y_train, pred_train), 4),
#     "cv_score":    round(gs.best_score_, 4),
#     "test_acc":    round(accuracy_score(y_test, pred_test), 4),
#     "test_auc":    round(roc_auc_score(y_test, proba_test), 4),
#     "test_logloss":round(log_loss(y_test, proba_test), 4),
# })


## 3.1 Logistic Regression

### How It Works
A linear model that predicts **probabilities** via the sigmoid function: $P(y=1|x) = \sigma(w^Tx + b) = \frac{1}{1 + e^{-(w^Tx + b)}}$.

Despite its name, it's a **classifier**, not a regression. The "regression" is on the **log-odds** (logit): $\log\frac{P(y=1)}{P(y=0)} = w^Tx + b$. This is a linear model in log-odds space.

Trained by maximizing log-likelihood (= minimizing binary cross-entropy): $-\sum[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)]$. No closed-form solution — uses iterative optimization (Newton-Raphson, L-BFGS, or coordinate descent for L1).

### When to Use / What Good Performance Means
- **Baseline for every classification task.** If it matches complex models → classes are linearly separable.
- Outputs **well-calibrated probabilities** out of the box — critical when you need $P(\text{event})$, not just the predicted class.
- **If LogReg ≈ GBM** → keep LogReg (simpler, more interpretable, faster, less overfit risk). Very common outcome in finance.

### Interview Edge — Quant Trading
- **Coefficients = log-odds ratios.** $\exp(\beta_i)$ = multiplicative change in odds for a 1-unit increase in feature $x_i$. Know this transformation cold.
- **Maximum likelihood can fail** with perfect/quasi-complete separation (coefficients → ∞). Regularization (L2) fixes this. Check for convergence warnings.
- **In quant:** regularized logistic regression is the industry standard for credit scoring (Basel regulations prefer interpretable models). For alpha signals, it's the simplest directional predictor (up/down).
- **Multiclass:** `multinomial` loss (softmax) vs `ovr` (one-vs-rest). Multinomial is more principled; OvR is simpler and sometimes works better.
- **Feature interactions are NOT captured** — you must engineer them explicitly (e.g., $x_1 \times x_2$).

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `C` | Inverse of regularization strength. LOWER C = MORE regularization | 0.001–100 (log scale) | **Opposite convention from SVM in sklearn!** C=0.01 = strong regularization |
| `penalty` | L1, L2, elasticnet, none | L2 default. L1 for feature selection | L1 requires solver='saga' or 'liblinear' |
| `class_weight` | Handles class imbalance via loss reweighting | 'balanced' or manual dict | 'balanced' auto-adjusts. But tuning threshold on ROC curve is often better |
| `solver` | Optimization algorithm | 'lbfgs' (default), 'saga' (L1/elasticnet), 'liblinear' (small data) | 'saga' needed for L1/elasticnet |
| `max_iter` | Max optimization iterations | 100 (default)–10000 | Increase if convergence warning appears |


In [ ]:
from sklearn.linear_model import LogisticRegression

model_configs = {
    "logreg_l2_all": {
        "model": LogisticRegression(max_iter=5000, solver="lbfgs", random_state=42),
        "param_grid": {
            "model__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    },
    "logreg_l1_all": {
        "model": LogisticRegression(penalty="l1", solver="saga", max_iter=5000, random_state=42),
        "param_grid": {
            "model__C": [0.001, 0.01, 0.1, 1.0, 10.0]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    },
}


In [ ]:
# ── Specific Methods to Know ──

# 1. Coefficients as odds ratios
best_lr = best_estimators.get("logreg_l2_all")
# feature_names = best_lr.named_steps["preprocessor"].get_feature_names_out()
# coefs = best_lr.named_steps["model"].coef_[0]
# odds_ratios = np.exp(coefs)
# pd.DataFrame({"feature": feature_names, "coef": coefs, "odds_ratio": odds_ratios})\
#     .sort_values("coef", key=abs, ascending=False).head(10)

# 2. Probability calibration check
# from sklearn.calibration import calibration_curve
# prob_true, prob_pred = calibration_curve(y_test, proba_test, n_bins=10)
# plt.plot(prob_pred, prob_true, "s-"); plt.plot([0,1],[0,1],"--"); plt.title("Calibration")

# 3. Threshold tuning (don't just use 0.5!)
# from sklearn.metrics import precision_recall_curve
# precisions, recalls, thresholds = precision_recall_curve(y_test, proba_test)
# # Find threshold that maximizes F1, or desired precision/recall tradeoff

# 4. ROC curve
# from sklearn.metrics import RocCurveDisplay
# RocCurveDisplay.from_estimator(best_lr, test_df[all_features], y_test)

# 5. Decision function vs predict_proba
# decision = best_lr.named_steps["model"].decision_function(X_test_processed)  # raw logits
# proba = best_lr.named_steps["model"].predict_proba(X_test_processed)  # calibrated probs


## 3.2 SVM (Support Vector Classifier)

### How It Works
Finds the **maximum-margin hyperplane** separating classes. The margin is the distance between the hyperplane and the closest points from each class (the support vectors). For non-linear boundaries, the **kernel trick** maps data to a higher-dimensional space where a linear separator exists.

**Hinge loss:** $\max(0, 1 - y \cdot f(x))$. Points correctly classified with margin > 1 have zero loss. Points inside the margin or misclassified are penalized linearly.

**Soft margin (C):** allows some violations of the margin. C controls the tradeoff: high C → narrow margin, fewer violations; low C → wide margin, more violations.

### When to Use / What Good Performance Means
- Excellent for clean, medium-sized datasets with a clear **margin of separation**.
- **Linear SVM ≈ logistic regression** in most cases, but with hinge loss instead of log-loss.
- **RBF SVM >> linear** → non-linear decision boundary exists.

### Interview Edge — Quant Trading
- **Margin maximization = structural risk minimization.** Know the connection to VC dimension and generalization bounds.
- **Primal vs dual formulation:** primal works in weight space (efficient when n >> d); dual works in data space (efficient when d >> n, and required for kernels).
- **Hinge loss vs log-loss:** hinge creates sparse solutions (only SVs matter); log-loss uses all points. Hinge is not differentiable at 1.
- **SVM probabilities (Platt scaling) are post-hoc** and often poorly calibrated. If you need real probabilities, use LogReg or calibrate explicitly.
- **Doesn't scale:** O(n²)–O(n³). In production quant, rarely used on large datasets. LinearSVC is scalable for the linear case.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `C` | Regularization. Higher = tighter fit, smaller margin | 0.01–1000 (log scale) | Soft margin: C controls margin width vs misclassification |
| `kernel` | Feature mapping. Defines model complexity | linear, rbf, poly(2-3) | "When in doubt, use RBF" — but try linear first |
| `gamma` (RBF) | Inverse influence radius. High = spiky = overfit | 'scale' or 1e-4 to 1e+1 | gamma too high → 100% train acc, garbage test acc |
| `class_weight` | Imbalance handling | 'balanced' or dict | Same as logistic regression |


In [ ]:
from sklearn.svm import SVC

model_configs = {
    "svc_rbf_all": {
        "model": SVC(kernel="rbf", probability=True, random_state=42),
        "param_grid": {
            "model__C": [0.1, 1.0, 10.0, 100.0],
            "model__gamma": ["scale", 0.001, 0.01, 0.1]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    },
    "svc_linear_all": {
        "model": SVC(kernel="linear", probability=True, random_state=42),
        "param_grid": {
            "model__C": [0.01, 0.1, 1.0, 10.0, 100.0]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    }
}


In [ ]:
# ── Specific Methods to Know ──

# 1. Support vectors analysis
best_svc = best_estimators.get("svc_rbf_all")
# n_sv = best_svc.named_steps["model"].n_support_  # per class
# sv_indices = best_svc.named_steps["model"].support_
# print(f"Support vectors per class: {n_sv}")
# High SV ratio → complex boundary; low → clean separation

# 2. Decision function (distance to hyperplane)
# distances = best_svc.named_steps["model"].decision_function(X_test_processed)
# Points close to 0 are near the boundary → most uncertain

# 3. For large datasets: use LinearSVC (much faster, no kernel)
from sklearn.svm import LinearSVC
# linear_svc = LinearSVC(C=1.0, max_iter=10000)
# Note: LinearSVC does NOT have predict_proba — calibrate with CalibratedClassifierCV
from sklearn.calibration import CalibratedClassifierCV
# calibrated = CalibratedClassifierCV(LinearSVC(C=1.0), cv=5)

# 4. Visualize decision boundary (2D only)
# from sklearn.inspection import DecisionBoundaryDisplay
# DecisionBoundaryDisplay.from_estimator(best_svc, X_2d, response_method="predict")


## 3.3 Random Forest / GBM Classifiers

### How It Works
Identical architectures to their regression counterparts, but with **classification-specific loss functions**:
- **RF:** Each tree uses **Gini impurity** or **entropy** to find splits. Final prediction = **majority vote** (or averaged probabilities) across all trees.
- **GBM:** Optimizes **log-loss** (binary cross-entropy) by default. Each tree is fit to the gradient of the log-loss.

### When to Use / What Good Performance Means
- **Top performers on tabular classification.** GBM (XGBoost/LightGBM) usually wins.
- Same bias-variance logic as regression: RF reduces variance; GBM reduces bias.
- If these models ≈ logistic regression → your problem is mostly linear. Keep LogReg.

### Interview Edge — Quant Trading
- **GBM with log-loss gives better probability calibration** than GBM with Gini. Always use log-loss for the objective if you need probabilities (which you almost always do in finance).
- **Class imbalance:** use `scale_pos_weight` (XGBoost) or `is_unbalance` (LightGBM) rather than SMOTE/oversampling. Resampling amplifies noise; reweighting doesn't.
- **RF probabilities are biased toward 0.5** (averaging binary predictions). GBM probabilities are usually better. Calibrate RF with `CalibratedClassifierCV` (isotonic regression) if needed.
- **In quant:** GBM classifiers for event prediction (earnings surprise direction, default/no-default, regime up/down). RF for stable regime classification.
- **Threshold tuning is critical for imbalanced data.** Default 0.5 is almost never optimal. Tune on precision-recall curve for your business objective.

### Key Parameters
All regression hyperparameters apply (see Section 2.3 and 2.4). Additional classification-specific:

| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `class_weight` (RF) | Reweights loss per class | 'balanced' or dict | Auto-adjusts for imbalance |
| `scale_pos_weight` (XGB) | Weight of positive class | neg_count / pos_count | More principled than resampling |
| `is_unbalance` (LGBM) | Auto-balances classes | True/False | Alternative to manual weight setting |
| Decision threshold | Post-model: which proba cutoff for class 1 | 0.1–0.9. Tune on val set | Default 0.5 is rarely optimal for imbalanced data |


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

model_configs = {
    "rf_clf_all": {
        "model": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        "param_grid": {
            "model__max_depth": [8, 12, 20, None],
            "model__max_features": ["sqrt", 0.3, 0.5],
            "model__min_samples_leaf": [1, 5, 10],
            "model__class_weight": [None, "balanced"]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    },
    "gbm_clf_all": {
        "model": GradientBoostingClassifier(random_state=42),
        "param_grid": {
            "model__n_estimators": [200, 500],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [3, 5, 7],
            "model__subsample": [0.7, 0.9]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    }
}


In [ ]:
# ── Specific Methods to Know ──

# 1. Confusion matrix heatmap
# from sklearn.metrics import ConfusionMatrixDisplay
# ConfusionMatrixDisplay.from_estimator(best_clf, test_df[all_features], y_test, cmap="Blues")

# 2. Classification report (precision, recall, F1 per class)
# print(classification_report(y_test, pred_test))

# 3. Threshold tuning on ROC curve
# from sklearn.metrics import roc_curve
# fpr, tpr, thresholds = roc_curve(y_test, proba_test)
# optimal_idx = np.argmax(tpr - fpr)  # Youden's J statistic
# optimal_threshold = thresholds[optimal_idx]
# print(f"Optimal threshold: {optimal_threshold:.3f}")

# 4. Probability calibration (especially for RF)
# from sklearn.calibration import CalibratedClassifierCV
# calibrated_rf = CalibratedClassifierCV(best_rf_clf, method="isotonic", cv=5)

# 5. Feature importance with SHAP
# import shap
# explainer = shap.TreeExplainer(best_clf.named_steps["model"])
# shap_values = explainer.shap_values(X_test_processed)


## 3.4 Naive Bayes

### How It Works
Applies **Bayes' theorem** with the "naive" assumption that features are **conditionally independent** given the class:

$$P(y|x_1,...,x_n) \propto P(y) \prod_{i=1}^{n} P(x_i|y)$$

Each feature's likelihood $P(x_i|y)$ is modeled separately:
- **GaussianNB:** assumes $P(x_i|y) \sim \mathcal{N}(\mu_{iy}, \sigma^2_{iy})$ — for continuous features
- **MultinomialNB:** assumes counts/frequencies — for text (bag of words, TF-IDF)
- **BernoulliNB:** assumes binary features — for binary text features

### When to Use / What Good Performance Means
- **Extremely fast** — both training and prediction are O(n·d). Works well with small data and high-dimensional sparse features.
- If NB works well → features are approximately independent given the class, OR the independence violation doesn't hurt prediction direction.
- **Competitive with complex models on text classification** — a classic result that surprises people.

### Interview Edge — Quant Trading
- **Independence assumption is almost always wrong** — but NB often works well anyway for prediction. It gets the ranking right even when probabilities are miscalibrated.
- **Probabilities are notoriously miscalibrated** (pushed toward 0 and 1). Always calibrate if you need actual probability estimates.
- **NB is a linear classifier in log-space.** $\log P(y|x) = \log P(y) + \sum \log P(x_i|y) + \text{const}$ — this is linear in log-likelihood features. Common interview surprise.
- **Zero-frequency problem:** if a feature value never appears with a class in training, $P(x_i|y) = 0$ → entire product = 0. Laplace smoothing (alpha) fixes this.
- **In quant:** rarely used for core alpha, but useful for NLP-based sentiment analysis on news articles, earnings call transcripts, SEC filings.
- **Generative vs discriminative:** NB is generative (models $P(x|y)$); LogReg is discriminative (models $P(y|x)$). With enough data, discriminative usually wins. With small data, generative can be better.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `alpha` (Laplace smoothing) | Additive smoothing. Prevents zero probabilities | 0.01–10. Default 1.0 | Lower for large data, higher for small/sparse. alpha=0 → crash on unseen values |
| `var_smoothing` (GaussianNB) | Added to variance estimate. Stabilizes near-zero variance | 1e-9 (default) to 1e-3 | Rarely tuned; more important to pick the right NB variant |
| `fit_prior` | Whether to learn class priors from data | True (default) | Set False for uniform priors |


In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB

model_configs = {
    "gnb_all": {
        "model": GaussianNB(),
        "param_grid": {
            "model__var_smoothing": [1e-11, 1e-9, 1e-7, 1e-5, 1e-3]
        },
        "features": all_features,
        "preprocessor": preprocessor_all
    },
    # MultinomialNB — for text/count data (requires non-negative features!)
    # "mnb_all": {
    #     "model": MultinomialNB(),
    #     "param_grid": {"model__alpha": [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]},
    #     "features": all_features,
    #     "preprocessor": preprocessor_all,  # NOTE: needs MinMaxScaler, not StandardScaler
    # }
}


In [ ]:
# ── Specific Methods to Know ──

# 1. Class log-probabilities (the core of NB predictions)
best_nb = best_estimators.get("gnb_all")
# log_probs = best_nb.named_steps["model"].predict_log_proba(X_test_processed[:5])
# proba = best_nb.named_steps["model"].predict_proba(X_test_processed[:5])

# 2. Learned parameters
# GaussianNB: theta_ (means per class), var_ (variance per class), class_prior_
# gnb = best_nb.named_steps["model"]
# print(f"Class priors: {gnb.class_prior_}")
# print(f"Feature means per class: {gnb.theta_.shape}")  # (n_classes, n_features)

# 3. Calibration (critical for NB — probabilities are unreliable raw)
from sklearn.calibration import CalibratedClassifierCV
# calibrated_nb = CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5)
# calibrated_nb.fit(X_train_processed, y_train)
# calibrated_proba = calibrated_nb.predict_proba(X_test_processed)

# 4. NB for text classification (the classic use case)
# from sklearn.feature_extraction.text import TfidfVectorizer
# tfidf = TfidfVectorizer(max_features=5000)
# X_text = tfidf.fit_transform(documents)
# mnb = MultinomialNB(alpha=1.0).fit(X_text, labels)


---
# 4. Unsupervised Learning Models

## Different Evaluation Paradigm

Unsupervised learning has **no target variable**, so the supervised skeleton (train/test split on y, MAE, R², accuracy) doesn't apply. Instead:

### Code Structure Differences
1. **No y_train / y_test.** You only have X.
2. **No GridSearchCV with scoring on y.** You evaluate with internal metrics:
   - **Silhouette Score** (-1 to 1): how similar a point is to its own cluster vs. nearest neighbor cluster. Higher = better. > 0.5 is good, > 0.7 is strong.
   - **Calinski-Harabasz Index**: ratio of between-cluster to within-cluster variance. Higher = better.
   - **Davies-Bouldin Index**: average similarity between each cluster and its most similar. Lower = better.
   - **Inertia** (K-Means specific): within-cluster sum of squares. Lower = better, but always decreases with more clusters (use elbow method).
   - **BIC/AIC** (GMM specific): penalized likelihood. Lower = better.
3. **No train/test prediction loop.** Fit on all data (or train, then transform test for dimensionality reduction).
4. **Validation = visual inspection + domain knowledge + metrics above.**

### General Skeleton for Unsupervised


In [ ]:
# ═══════════════════════════════════════════════════════
# UNSUPERVISED EVALUATION SKELETON
# ═══════════════════════════════════════════════════════

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# ── 1. Preprocess (always standardize for distance-based methods) ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train[numeric_features])
# For mixed data, use preprocessor_all.fit_transform(X_train)

# ── 2. Generic clustering evaluation function ──
def evaluate_clustering(model, X, name=""):
    """Evaluate a fitted clustering model with standard metrics."""
    labels = model.labels_ if hasattr(model, "labels_") else model.predict(X)
    n_clusters = len(set(labels) - {-1})  # exclude noise label (-1)
    n_noise = (labels == -1).sum()

    metrics = {"name": name, "n_clusters": n_clusters, "n_noise": n_noise}
    if n_clusters >= 2:
        mask = labels != -1  # exclude noise for metrics
        metrics["silhouette"]      = round(silhouette_score(X[mask], labels[mask]), 4)
        metrics["calinski"]        = round(calinski_harabasz_score(X[mask], labels[mask]), 1)
        metrics["davies_bouldin"]  = round(davies_bouldin_score(X[mask], labels[mask]), 4)
    return metrics

# ── 3. Example sweep: K-Means over K ──
from sklearn.cluster import KMeans

results = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    res = evaluate_clustering(km, X_scaled, name=f"kmeans_k{k}")
    res["inertia"] = round(km.inertia_, 1)
    results.append(res)

results_df = pd.DataFrame(results)
display(results_df)

# ── 4. Elbow & silhouette plots ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(results_df["n_clusters"], results_df["inertia"], "o-")
axes[0].set_title("Elbow: Inertia"); axes[0].set_xlabel("K")
axes[1].plot(results_df["n_clusters"], results_df["silhouette"], "o-")
axes[1].set_title("Silhouette Score"); axes[1].set_xlabel("K")
axes[2].plot(results_df["n_clusters"], results_df["davies_bouldin"], "o-")
axes[2].set_title("Davies-Bouldin (lower=better)"); axes[2].set_xlabel("K")
plt.tight_layout(); plt.show()


## 4.1 K-Means Clustering

### How It Works
1. Initialize K centroids (K-Means++ for smart init)
2. **E-step:** Assign each point to the nearest centroid
3. **M-step:** Update centroids to the mean of assigned points
4. Repeat until convergence (assignments don't change)

Minimizes **within-cluster sum of squares** (inertia): $\sum_{k=1}^{K} \sum_{x \in C_k} ||x - \mu_k||^2$

### When to Use / What Good Performance Means
- Fast and scalable. Best for **roughly spherical, equal-size clusters** with clear separation.
- High silhouette score (>0.5) → clusters are well-separated and cohesive.
- In quant: **regime detection** (market states), client segmentation, grouping correlated assets.

### Interview Edge — Quant Trading
- **Assumes spherical, equal-size clusters** (isotropic Gaussian). Fails on elongated or variable-density clusters → use GMM instead.
- **Result depends on initialization** → always use K-Means++ (default) and run multiple inits (n_init=10+).
- **How to choose K:** elbow method (inertia vs K), silhouette score, gap statistic. None is definitive — **domain knowledge matters most**. In finance: 2–4 regimes is standard.
- **K-Means is equivalent to GMM** with equal, spherical covariances and hard assignments. Understanding this equivalence impresses interviewers.
- **In quant:** K-Means on returns for regime detection is common but fragile. Use **rolling windows + stability analysis** to validate. Cluster labels can flip between runs — track by centroid properties, not label numbers.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `n_clusters` (K) | The fundamental choice | 2–20. Use elbow/silhouette. Finance: 2–4 | Wrong K renders results meaningless |
| `init` | Initialization strategy | Always use 'k-means++' | K-means++ gives O(log K) competitive approximation |
| `n_init` | Number of random restarts | 10 (default). 20–50 for critical work | Cheap insurance against bad local minima |
| `max_iter` | Max iterations per run | 300 (default). Increase if not converging | Rarely needs tuning |


In [ ]:
from sklearn.cluster import KMeans

# ── Parameter sweep for K-Means ──
kmeans_configs = {}
for k in [2, 3, 4, 5, 6, 8, 10]:
    km = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=42)
    km.fit(X_scaled)
    kmeans_configs[f"kmeans_k{k}"] = {
        "model": km,
        "metrics": evaluate_clustering(km, X_scaled, f"k={k}"),
        "inertia": km.inertia_,
    }

# Display results
pd.DataFrame([v["metrics"] for v in kmeans_configs.values()])


In [ ]:
# ── Specific Methods to Know ──

# 1. Cluster centers (centroids) — the core output
best_km = kmeans_configs["kmeans_k3"]["model"]
centroids = best_km.cluster_centers_
print(f"Centroid shape: {centroids.shape}")  # (K, n_features)
# In finance: centroids represent "average regime profile"

# 2. Labels and transform (distance to each centroid)
labels = best_km.labels_
distances = best_km.transform(X_scaled)  # (n_samples, K) — distance to each centroid
# Useful for soft assignments or as features

# 3. Silhouette analysis (per-sample)
from sklearn.metrics import silhouette_samples
# sil_samples = silhouette_samples(X_scaled, labels)
# # Plot silhouette per cluster to identify weak clusters

# 4. Mini-batch K-Means (for large datasets)
from sklearn.cluster import MiniBatchKMeans
# mb_km = MiniBatchKMeans(n_clusters=4, batch_size=1000, random_state=42)
# mb_km.fit(X_scaled)  # Much faster, slight quality tradeoff

# 5. Predict new data
# new_labels = best_km.predict(X_test_scaled)  # assign new points to existing clusters


## 4.2 Gaussian Mixture Model (GMM)

### How It Works
Assumes data is generated from a **mixture of K Gaussian distributions**, each with its own mean $\mu_k$, covariance $\Sigma_k$, and mixing weight $\pi_k$:

$$P(x) = \sum_{k=1}^{K} \pi_k \cdot \mathcal{N}(x | \mu_k, \Sigma_k)$$

Fit via the **EM algorithm**:
1. **E-step:** Compute soft assignments — probability each point belongs to each component: $\gamma_{ik} = P(z_i = k | x_i)$
2. **M-step:** Update $\mu_k, \Sigma_k, \pi_k$ using the soft assignments
3. Repeat until log-likelihood converges

### When to Use / What Good Performance Means
- **Generalizes K-Means** to clusters with different shapes, sizes, and orientations.
- Provides **soft assignments** (probabilities), not just hard labels — much more useful in practice.
- If GMM >> K-Means → clusters are non-spherical or have different variances.

### Interview Edge — Quant Trading
- **EM converges to a LOCAL maximum** — initialization matters. Use K-Means initialization.
- **BIC for model selection:** $BIC = -2\log L + p \log n$. Lower = better. BIC penalizes complexity more than AIC → preferred for finite samples.
- **Covariance type matters enormously:** 'full' is most flexible (K × d² params) but overfits in high-D; 'diag' assumes independence (K × d params); 'spherical' reduces to K-Means.
- **Singularity problem:** if a Gaussian collapses onto a single point, likelihood → ∞. `reg_covar` prevents this.
- **In quant:** GMM on returns for regime detection gives **probabilistic regime membership** — you can say "we're 70% in the low-vol regime." Much more useful than hard K-Means labels. Also used for VaR estimation and tail modeling.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `n_components` | Number of Gaussians | 2–10. Use BIC to select | Finance regime detection: 2–3 typical |
| `covariance_type` | Shape flexibility | 'full', 'tied', 'diag', 'spherical' | 'full' if n >> d. 'diag' if many features |
| `reg_covar` | Regularization on covariance | 1e-6 (default). Increase to 1e-3 if issues | Essential safety valve against singularity |
| `n_init` | Number of EM restarts | 1–10 | Multiple inits reduce local optima risk |


In [ ]:
from sklearn.mixture import GaussianMixture

# ── BIC/AIC sweep for number of components ──
gmm_results = []
for n in range(2, 11):
    for cov_type in ["full", "diag", "tied"]:
        gmm = GaussianMixture(n_components=n, covariance_type=cov_type,
                               n_init=5, random_state=42, reg_covar=1e-5)
        gmm.fit(X_scaled)
        labels = gmm.predict(X_scaled)
        sil = silhouette_score(X_scaled, labels) if len(set(labels)) >= 2 else -1
        gmm_results.append({
            "n_components": n, "cov_type": cov_type,
            "BIC": round(gmm.bic(X_scaled), 1),
            "AIC": round(gmm.aic(X_scaled), 1),
            "silhouette": round(sil, 4),
        })

gmm_df = pd.DataFrame(gmm_results)
print("Best by BIC:")
display(gmm_df.sort_values("BIC").head(5))


In [ ]:
# ── Specific Methods to Know ──

# 1. Soft assignments (probabilities) — GMM's key advantage over K-Means
best_gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=42)
best_gmm.fit(X_scaled)
proba = best_gmm.predict_proba(X_scaled)  # (n_samples, n_components)
# proba[i] = [P(cluster 0), P(cluster 1), P(cluster 2)] — sums to 1
# In finance: "sample i has 70% chance of being in regime 0"

# 2. Per-sample log-likelihood (anomaly detection)
log_lik = best_gmm.score_samples(X_scaled)  # per-sample log-likelihood
# Low log-likelihood → anomaly/outlier

# 3. Learned parameters
# best_gmm.means_          # (K, d) — component means
# best_gmm.covariances_    # (K, d, d) for 'full' — component covariances
# best_gmm.weights_         # (K,) — mixing proportions

# 4. BIC plot for component selection
# bics = [GaussianMixture(n_components=k).fit(X_scaled).bic(X_scaled) for k in range(1, 11)]
# plt.plot(range(1, 11), bics, "o-"); plt.xlabel("K"); plt.title("BIC")

# 5. Generate synthetic samples
# samples, labels = best_gmm.sample(n_samples=100)


## 4.3 DBSCAN (Density-Based Spatial Clustering)

### How It Works
1. For each point, count how many neighbors are within radius ε.
2. If a point has ≥ `min_samples` neighbors → it's a **core point**.
3. Core points within ε of each other are connected into the same cluster.
4. **Border points** are within ε of a core point but don't have enough neighbors themselves.
5. Points not reachable from any core point are labeled as **noise** (label = -1).

No need to specify K — DBSCAN discovers the number of clusters automatically.

### When to Use / What Good Performance Means
- When clusters have **arbitrary shapes** (not just spherical) and you need **automatic outlier detection**.
- Doesn't require specifying K — discovers it from the data.
- If DBSCAN finds meaningful clusters → your data has clear density-separated regions.

### Interview Edge — Quant Trading
- **Fails with variable-density clusters:** one cluster's core region is another's noise. **HDBSCAN** fixes this — mention it proactively.
- **Choosing ε:** use the **k-distance plot** (sort distances to k-th nearest neighbor, find the elbow). k = min_samples.
- **Not suitable for high-dimensional data** — distances lose meaning (curse of dimensionality). Reduce dimensions first (PCA, UMAP).
- **In quant:** anomaly detection (unusual trading patterns, extreme returns), identifying market microstructure regimes, fraud detection.
- **Deterministic** (unlike K-Means) — given ε and min_samples, you always get the same result.
- **Noise points (label = -1)** are a feature, not a bug. They represent genuine outliers.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `eps` (ε) | Neighborhood radius. Too small → all noise. Too large → one cluster | Use k-distance plot. Standardize features first! | THE critical parameter. Encodes "close" |
| `min_samples` | Min points for a core point | 2×d rule of thumb. Min 3. Typically 5–20 | Higher = more conservative (more noise) |
| `metric` | Distance function | 'euclidean' (default), 'manhattan', 'cosine' | Cosine for text/normalized data |


In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# ── Step 1: k-distance plot to find eps ──
min_samples_val = 5  # start with 2 * n_dimensions or 5
nn = NearestNeighbors(n_neighbors=min_samples_val)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])  # distance to k-th neighbor

plt.figure(figsize=(10, 4))
plt.plot(k_distances)
plt.xlabel("Points (sorted)"); plt.ylabel(f"{min_samples_val}-th neighbor distance")
plt.title("k-Distance Plot — look for the elbow")
plt.show()

# ── Step 2: sweep eps around the elbow ──
dbscan_results = []
for eps_val in np.arange(0.3, 2.0, 0.1):
    db = DBSCAN(eps=eps_val, min_samples=min_samples_val)
    db.fit(X_scaled)
    labels = db.labels_
    n_clusters = len(set(labels) - {-1})
    n_noise = (labels == -1).sum()
    sil = silhouette_score(X_scaled, labels) if n_clusters >= 2 and n_noise < len(labels) - 2 else -1
    dbscan_results.append({
        "eps": round(eps_val, 2), "n_clusters": n_clusters,
        "n_noise": n_noise, "noise_pct": round(n_noise / len(labels) * 100, 1),
        "silhouette": round(sil, 4),
    })

display(pd.DataFrame(dbscan_results))


In [ ]:
# ── Specific Methods to Know ──

# 1. Core, border, and noise points
best_db = DBSCAN(eps=0.8, min_samples=5).fit(X_scaled)
labels = best_db.labels_
core_mask = np.zeros(len(labels), dtype=bool)
core_mask[best_db.core_sample_indices_] = True
noise_mask = labels == -1
print(f"Core: {core_mask.sum()}, Border: {(~core_mask & ~noise_mask).sum()}, Noise: {noise_mask.sum()}")

# 2. DBSCAN has NO predict method — it's transductive
# For new points: train a KNN classifier on DBSCAN labels
# from sklearn.neighbors import KNeighborsClassifier
# knn_proxy = KNeighborsClassifier(n_neighbors=5).fit(X_scaled[~noise_mask], labels[~noise_mask])
# new_labels = knn_proxy.predict(X_new_scaled)

# 3. HDBSCAN — the improved version (handles variable density)
# pip install hdbscan
# import hdbscan
# clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5)
# clusterer.fit(X_scaled)
# clusterer.probabilities_  # soft cluster membership

# 4. Anomaly scores from DBSCAN
# Use distance to nearest core point as anomaly score
# Or use HDBSCAN's outlier_scores_


## 4.4 PCA (Principal Component Analysis)

### How It Works
Finds the **orthogonal directions** (principal components) along which data varies the most:
1. Center the data (subtract mean)
2. Compute the covariance matrix $C = \frac{1}{n-1}X^TX$
3. Eigendecompose: $C = V \Lambda V^T$ where columns of V are the principal components
4. Project data onto the top K eigenvectors (those with largest eigenvalues)

Equivalently: PCA finds the K-dimensional subspace that maximizes the variance of the projected data (= minimizes reconstruction error).

### When to Use / What Good Performance Means
- **Dimensionality reduction** before modeling, visualization, denoising.
- If top few PCs explain >80% variance → data lies on a low-dimensional manifold.
- **In quant:** factor models (statistical arbitrage), risk decomposition, denoising covariance matrices.

### Interview Edge — Quant Trading
- **ALWAYS standardize first** (unless features are on the same scale). PCA maximizes variance — unscaled features with large values will dominate.
- **PCs maximize variance, NOT predictive power.** High-variance directions may be noise. For supervised dimensionality reduction, use **PLS** (Partial Least Squares) instead.
- **First PC on stock returns ≈ "market factor."** This IS the foundation of statistical arbitrage. Residuals after removing PCA factors are the "alpha" component.
- **Marchenko-Pastur law:** for a random matrix of size n×d, eigenvalues follow the MP distribution. Eigenvalues ABOVE the MP bound are likely real factors; below = noise. More principled than the scree plot. Key paper for quant interviews.
- **Ledoit-Wolf shrinkage** for covariance estimation improves PCA stability on financial data.
- **PCA on the correlation matrix of returns → statistical risk factors.** Number of significant PCs ≈ number of independent risk factors.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `n_components` | Components to keep | Float (0.95 for 95% var) or int (3–15 for equity factors) | Use scree plot or Marchenko-Pastur |
| `whiten` | Scale components to unit variance | False (default). True for distance-based downstream | Removes scale info |
| `svd_solver` | Algorithm | 'auto', 'full', 'randomized' | 'randomized' much faster for large data |


In [ ]:
from sklearn.decomposition import PCA

# ── Full PCA for analysis ──
pca_full = PCA()
pca_full.fit(X_scaled)

# Explained variance analysis
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_for_90 = np.argmax(cumvar >= 0.90) + 1
n_for_95 = np.argmax(cumvar >= 0.95) + 1
print(f"Components for 90% variance: {n_for_90}")
print(f"Components for 95% variance: {n_for_95}")

# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, min(21, len(pca_full.explained_variance_ratio_) + 1)),
            pca_full.explained_variance_ratio_[:20])
axes[0].set_title("Scree Plot (individual)"); axes[0].set_xlabel("PC")
axes[1].plot(range(1, min(21, len(cumvar) + 1)), cumvar[:20], "o-")
axes[1].axhline(0.90, ls="--", color="r", label="90%")
axes[1].axhline(0.95, ls="--", color="g", label="95%")
axes[1].set_title("Cumulative Variance"); axes[1].legend()
plt.tight_layout(); plt.show()

# ── PCA with chosen n_components for downstream use ──
pca = PCA(n_components=0.95, svd_solver="full")  # auto-select for 95%
X_pca = pca.fit_transform(X_scaled)
print(f"Reduced: {X_scaled.shape[1]} features -> {X_pca.shape[1]} components")


In [ ]:
# ── Specific Methods to Know ──

# 1. Component loadings (what each PC means)
loadings = pd.DataFrame(
    pca.components_.T,
    index=numeric_features[:pca.components_.shape[1]] if len(numeric_features) >= pca.components_.shape[1] else range(pca.components_.shape[1]),
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)
# Top features for PC1:
# print(loadings["PC1"].abs().sort_values(ascending=False).head(10))

# 2. Inverse transform (reconstruction) — anomaly detection
X_reconstructed = pca.inverse_transform(X_pca)
recon_error = np.mean((X_scaled - X_reconstructed) ** 2, axis=1)
# High reconstruction error → anomaly (doesn't fit the learned structure)

# 3. Marchenko-Pastur threshold (finance-specific)
n, d = X_scaled.shape
sigma2 = 1.0  # if standardized
lambda_plus = sigma2 * (1 + np.sqrt(d / n)) ** 2
n_signal = (pca_full.explained_variance_ > lambda_plus).sum()
print(f"Marchenko-Pastur: {n_signal} significant components (threshold={lambda_plus:.4f})")

# 4. PCA as preprocessing in a pipeline
# pipe = Pipeline([
#     ("scaler", StandardScaler()),
#     ("pca", PCA(n_components=10)),
#     ("model", Ridge(alpha=1.0)),
# ])


## 4.5 Hierarchical Clustering (Agglomerative)

### How It Works
**Bottom-up (agglomerative):**
1. Start with each point as its own cluster
2. At each step, merge the two closest clusters
3. Repeat until all points are in one cluster
4. The result is a **dendrogram** — a tree of all merges

The **linkage method** defines "distance between clusters":
- **Ward:** minimize the increase in total within-cluster variance → equal-size, spherical clusters
- **Complete:** max distance between any pair → compact clusters
- **Average:** mean distance between all pairs → compromise
- **Single:** min distance between any pair → prone to chaining (long, thin clusters)

### When to Use / What Good Performance Means
- When you need a **hierarchy** at multiple granularity levels, not just one partition.
- When you don't know K and want to **explore** structure visually (dendrogram).
- In quant: **asset taxonomy**, hierarchical risk parity portfolios.

### Interview Edge — Quant Trading
- **Ward is usually the best default.** Single linkage is pathological (chaining effect).
- **O(n²) memory, O(n³) time.** Not scalable beyond ~10k points.
- **Hierarchical Risk Parity (HRP)** uses hierarchical clustering on the correlation matrix to build portfolios. Often beats Markowitz. Key paper: López de Prado, 2016.
- **Cophenetic correlation:** measures how well the dendrogram preserves original pairwise distances. > 0.7 is acceptable.
- **Ward requires Euclidean distance.** For financial data, use average/complete linkage with **correlation distance** (1 - ρ).

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `n_clusters` or `distance_threshold` | Where to cut the dendrogram | n_clusters: 2–20 | Use dendrogram to pick |
| `linkage` | Cluster distance definition | 'ward' (best default), 'complete', 'average'. Avoid 'single' | Ward needs Euclidean |
| `metric` | Point-to-point distance | 'euclidean' (default). Finance: 1-correlation | Correlation distance is gold standard for assets |


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, cophenet
from scipy.spatial.distance import pdist

# ── Step 1: Compute linkage and plot dendrogram ──
Z = linkage(X_scaled, method="ward", metric="euclidean")

plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=8)
plt.title("Dendrogram (Ward linkage)")
plt.xlabel("Cluster size"); plt.ylabel("Distance")
plt.tight_layout(); plt.show()

# ── Step 2: Cophenetic correlation ──
coph_corr, _ = cophenet(Z, pdist(X_scaled))
print(f"Cophenetic correlation: {coph_corr:.4f}")  # > 0.7 is acceptable

# ── Step 3: Cut at different levels and evaluate ──
hier_results = []
for k in range(2, 11):
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    hier_results.append({"n_clusters": k, "silhouette": round(sil, 4)})

display(pd.DataFrame(hier_results))


In [ ]:
# ── Specific Methods to Know ──

# 1. Dendrogram with color threshold
plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode="lastp", p=30, color_threshold=0.7 * max(Z[:, 2]))
plt.title("Dendrogram with color threshold")
plt.tight_layout(); plt.show()

# 2. Cut dendrogram at a specific distance
from scipy.cluster.hierarchy import fcluster
labels_by_dist = fcluster(Z, t=5.0, criterion="distance")
labels_by_k = fcluster(Z, t=4, criterion="maxclust")

# 3. Correlation-based clustering for financial data (HRP-style)
# corr_matrix = returns_df.corr()
# dist_matrix = np.sqrt(0.5 * (1 - corr_matrix))  # correlation distance
# from scipy.spatial.distance import squareform
# Z_corr = linkage(squareform(dist_matrix), method="average")

# 4. Compare linkage methods
for method in ["ward", "complete", "average", "single"]:
    Z_test = linkage(X_scaled, method=method)
    coph, _ = cophenet(Z_test, pdist(X_scaled))
    print(f"{method:>10}: cophenetic = {coph:.4f}")


## 4.6 t-SNE / UMAP (Visualization & Non-linear Dimensionality Reduction)

### How It Works
Both are **non-linear dimensionality reduction** methods primarily for **visualization** in 2D/3D.

**t-SNE (t-distributed Stochastic Neighbor Embedding):**
1. Compute pairwise similarities in high-D using Gaussian kernels → probability distribution P
2. Initialize random 2D layout
3. Define similarities in 2D using Student's t-distribution (heavier tails) → distribution Q
4. Minimize KL divergence between P and Q via gradient descent
5. The t-distribution's heavy tails prevent the "crowding problem"

**UMAP (Uniform Manifold Approximation and Projection):**
- Based on topological data analysis (Riemannian geometry)
- Constructs a fuzzy simplicial set in high-D and optimizes a low-D representation
- Preserves **more global structure** than t-SNE and is **much faster**

### When to Use / What Good Performance Means
- **Visualization only** (especially t-SNE). UMAP can also be used for general dimensionality reduction.
- If clear clusters appear → genuine structure exists. But **always verify quantitatively**.
- NOT for downstream modeling with t-SNE. UMAP embeddings can cautiously be used as features.

### Interview Edge — Quant Trading
- **t-SNE distances between clusters are MEANINGLESS.** Only within-cluster structure is reliable. Cluster sizes are also meaningless. This is the #1 misinterpretation — mentioning it unprompted shows expertise.
- **Different perplexity → completely different pictures.** ALWAYS try multiple values. Show you know this.
- **UMAP is generally preferred now:** faster, scales better, preserves more global structure.
- **Neither is deterministic** — run multiple times to check stability.
- **In quant:** useful for exploring feature spaces, visualizing market regimes. **Never use t-SNE as a feature for prediction.**
- **UMAP can transform new data** (`transform` method). t-SNE cannot.

### Key Parameters
| Parameter | Impact | Range | Nuance |
|---|---|---|---|
| `perplexity` (t-SNE) | Effective # neighbors. Local vs global | 5–50. ALWAYS try multiple | Low → local. High → global. Very sensitive |
| `n_neighbors` (UMAP) | Similar to perplexity | 5–200. Default 15 | Less sensitive than t-SNE perplexity |
| `min_dist` (UMAP) | Min distance in embedding | 0.0–0.99. Default 0.1 | 0 → tight clusters. Higher → spread |
| `n_iter` (t-SNE) | Optimization iterations | 1000–5000 | More = safer |
| `metric` | Distance for neighbors | euclidean, cosine, correlation | correlation for returns |


In [ ]:
from sklearn.manifold import TSNE
# from umap import UMAP  # pip install umap-learn

# ── t-SNE with multiple perplexity values (ALWAYS DO THIS) ──
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, perp in zip(axes, [5, 15, 30, 50]):
    tsne = TSNE(n_components=2, perplexity=perp, n_iter=1000, random_state=42)
    embedding = tsne.fit_transform(X_scaled)
    ax.scatter(embedding[:, 0], embedding[:, 1], s=5, alpha=0.5)
    ax.set_title(f"perplexity={perp}")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("t-SNE: ALWAYS compare multiple perplexity values")
plt.tight_layout(); plt.show()

# ── UMAP (preferred in practice) ──
# umap_model = UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
# embedding_umap = umap_model.fit_transform(X_scaled)
# plt.scatter(embedding_umap[:, 0], embedding_umap[:, 1], s=5, alpha=0.5)
# plt.title("UMAP"); plt.show()


In [ ]:
# ── Specific Methods to Know ──

# 1. Color by known labels or cluster assignments
# plt.scatter(embedding[:, 0], embedding[:, 1], c=labels, cmap="tab10", s=5)

# 2. t-SNE with precomputed distances (for custom metrics)
# from sklearn.metrics import pairwise_distances
# dist_matrix = pairwise_distances(X_scaled, metric="cosine")
# tsne = TSNE(metric="precomputed", perplexity=30)
# embedding = tsne.fit_transform(dist_matrix)

# 3. UMAP: fit on train, transform test (t-SNE can't do this!)
# umap_model = UMAP(n_neighbors=15, min_dist=0.1).fit(X_train_scaled)
# test_embedding = umap_model.transform(X_test_scaled)

# 4. UMAP as preprocessing for clustering (viable, unlike t-SNE)
# X_umap = UMAP(n_components=5).fit_transform(X_scaled)  # reduce to 5D
# km = KMeans(n_clusters=4).fit(X_umap)  # cluster in UMAP space

# 5. Stability check: run multiple times with different seeds
# embeddings = [TSNE(perplexity=30, random_state=seed).fit_transform(X_scaled)
#               for seed in range(5)]
# If structure is consistent across seeds → it's real


---
# Quick Reference: Model Selection Decision Tree

**Regression:**
1. Start with **Ridge** (baseline, handles multicollinearity)
2. Try **Lasso** if you suspect sparsity
3. Try **Random Forest** for non-linearity without much tuning
4. Try **GBM** (XGBoost/LightGBM) for maximum performance
5. **SVR** / **KNN** for small datasets or specific use cases

**Classification:**
1. Start with **Logistic Regression** (baseline, calibrated probabilities)
2. Try **SVM (RBF)** if you suspect non-linear boundary + medium data
3. Try **RF / GBM** for maximum performance on tabular data
4. **Naive Bayes** for text data or very fast baseline

**Unsupervised:**
1. **PCA** first for dimensionality reduction / understanding structure
2. **K-Means** for fast, spherical clustering
3. **GMM** for soft assignments / non-spherical clusters
4. **DBSCAN/HDBSCAN** for arbitrary shapes + outlier detection
5. **Hierarchical** for dendrogram analysis + HRP portfolios
6. **t-SNE/UMAP** for visualization only

**Cross-cutting interview tips:**
- Always discuss **bias-variance tradeoff**
- **Never use random CV on time-series** → TimeSeriesSplit or purged CV
- **Feature engineering > model choice** for alpha generation
- Be ready to discuss **stacking** (linear blend of model predictions)
- If asked "which model would you use?" → the answer is always "it depends" — then explain the tradeoffs
- Know the **computational complexity** of each model (training + prediction)
